# 09 — ETL Real World
**Goal:** Build a complete pipeline: raw Adobe + paid CSVs → clean unified table → charts.

```
Adobe CSV  →  extract  →  clean  →  transform  →
Paid CSV   →  extract  →  clean  →  aggregate  →  merge  →  unified  →  charts
```

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

Path('data/raw/samples').mkdir(parents=True, exist_ok=True)
Path('data/clean').mkdir(exist_ok=True)

## Step 0 — Generate realistic raw files

In [ ]:
np.random.seed(42)
dates = pd.date_range('2024-01-01', '2024-03-31')

adobe_rows = []
for d in dates:
    visits  = int(np.random.randint(2500, 5000) * (0.8 if d.weekday() >= 5 else 1.0))
    orders  = int(visits * np.random.uniform(0.01, 0.04))
    revenue = round(orders * np.random.uniform(300, 800), 2)
    adobe_rows.append({
        'Date':                 d.strftime('%m/%d/%Y'),
        'Visits':               f'{visits:,}',
        'Unique Visitors':      f'{int(visits*0.88):,}',
        'Page Views':           f'{int(visits*3.2):,}',
        'Bounce Rate':          f'{np.random.uniform(35,55):.1f}%',
        'Avg Time Spent (sec)': int(np.random.uniform(120, 320)),
        'Orders':               orders,
        'Revenue':              f'${revenue:,.2f}',
    })

adobe_df = pd.DataFrame(adobe_rows)
header = 'Report Suite: Fintech_CL\nDate Range: 01/01/2024 - 03/31/2024\nGenerated: 04/01/2024\n\n'
with open('data/raw/samples/adobe_q1_2024.csv', 'w') as f:
    f.write(header)
    adobe_df.to_csv(f, index=False)

print('Adobe file created:', adobe_df.shape)
adobe_df.head(3)

In [ ]:
campaigns = ['Brand_Search_CL', 'NonBrand_Search_CL', 'Remarketing_CL', 'Display_Prospecting_CL']
paid_rows = []
for d in dates:
    for camp in campaigns:
        impr   = int(np.random.randint(10000, 80000) * (0.7 if d.weekday() >= 5 else 1.0))
        clicks = int(impr * np.random.uniform(0.01, 0.05))
        cost   = round(clicks * np.random.uniform(0.3, 1.5), 2)
        conv   = int(clicks * np.random.uniform(0.01, 0.04))
        paid_rows.append({
            'Day':         d.strftime('%Y-%m-%d'),
            'Campaign':    camp,
            'Impressions': f'{impr:,}',
            'Clicks':      clicks,
            'Cost':        f'${cost:,.2f}',
            'Conversions': conv,
            'Conv. Value': f'${conv * np.random.uniform(400, 900):.2f}',
        })

paid_df = pd.DataFrame(paid_rows)
paid_df.to_csv('data/raw/samples/paid_q1_2024.csv', index=False)
print('Paid file created:', paid_df.shape)
paid_df.head(4)

## Step 1 — Extract & Clean Adobe

In [ ]:
def extract_adobe(path):
    df = pd.read_csv(path, skiprows=4, thousands=',')
    df = df.drop_duplicates().dropna(subset=['Date'])

    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y')

    df['Bounce Rate'] = (
        df['Bounce Rate'].str.replace('%', '', regex=False).astype(float) / 100
    )
    df = df[df['Bounce Rate'].between(0, 1)]

    df['Revenue'] = (
        df['Revenue'].str.replace('$', '', regex=False)
                     .str.replace(',', '', regex=False)
                     .astype(float).clip(lower=0)
    )

    df.columns = [
        'date', 'visits', 'unique_visitors', 'pageviews',
        'bounce_rate', 'avg_time_sec', 'orders', 'revenue'
    ]
    return df.sort_values('date').reset_index(drop=True)


adobe_clean = extract_adobe('data/raw/samples/adobe_q1_2024.csv')
print('Shape:', adobe_clean.shape)
print(adobe_clean.dtypes)
adobe_clean.head()

## Step 2 — Extract & Clean Paid

In [ ]:
def extract_paid(path):
    df = pd.read_csv(path, thousands=',')
    df = df.drop_duplicates().dropna(subset=['Day'])
    df['Day'] = pd.to_datetime(df['Day'])

    for col in ['Cost', 'Conv. Value']:
        df[col] = (
            df[col].str.replace('$', '', regex=False)
                   .str.replace(',', '', regex=False)
                   .astype(float).clip(lower=0)
        )

    df['Campaign'] = df['Campaign'].str.strip().str.lower()
    df['campaign_type'] = df['Campaign'].str.extract(r'(brand|nonbrand|remarketing|display)')[0]

    df.columns = [
        'date', 'campaign', 'impressions', 'clicks',
        'cost', 'conversions', 'conv_value', 'campaign_type'
    ]
    return df.sort_values(['date', 'campaign']).reset_index(drop=True)


paid_clean = extract_paid('data/raw/samples/paid_q1_2024.csv')
print('Shape:', paid_clean.shape)
paid_clean.head()

## Step 3 — Aggregate Paid to Daily

In [ ]:
def aggregate_paid_daily(df):
    daily = df.groupby('date').agg(
        total_spend       = ('cost',        'sum'),
        total_impressions = ('impressions',  'sum'),
        total_clicks      = ('clicks',       'sum'),
        paid_conversions  = ('conversions',  'sum'),
        conv_value        = ('conv_value',   'sum'),
    ).reset_index()

    daily = daily.assign(
        ctr  = lambda x: x['total_clicks'] / x['total_impressions'] * 100,
        cpc  = lambda x: x['total_spend'] / x['total_clicks'],
        cpa  = lambda x: x['total_spend'] / x['paid_conversions'],
        roas = lambda x: x['conv_value'] / x['total_spend'],
    )
    return daily


paid_daily = aggregate_paid_daily(paid_clean)
print('Shape:', paid_daily.shape)
paid_daily.head().round(2)

## Step 4 — Merge into Unified Table

In [ ]:
def build_unified(adobe, paid):
    unified = pd.merge(adobe, paid, on='date', how='left')

    paid_cols = ['total_spend','total_impressions','total_clicks','paid_conversions','conv_value']
    unified[paid_cols] = unified[paid_cols].fillna(0)

    unified = unified.assign(
        cvr               = lambda x: x['orders'] / x['visits'] * 100,
        paid_share        = lambda x: x['paid_conversions'] / x['orders'].replace(0, np.nan) * 100,
        revenue_per_visit = lambda x: x['revenue'] / x['visits'],
        month             = lambda x: x['date'].dt.month,
        week              = lambda x: x['date'].dt.isocalendar().week.astype(int),
        weekday           = lambda x: x['date'].dt.day_name(),
        is_weekend        = lambda x: x['date'].dt.dayofweek >= 5,
        wow_visits_pct    = lambda x: (x['visits'] - x['visits'].shift(7)) / x['visits'].shift(7) * 100,
    )
    return unified.sort_values('date').reset_index(drop=True)


unified = build_unified(adobe_clean, paid_daily)
print('Unified shape:', unified.shape)
print('Columns:', unified.columns.tolist())
unified[['date','visits','orders','cvr','total_spend','cpa','roas','wow_visits_pct']].head(10).round(2)

## Step 5 — Validate & Save

In [ ]:
def validate(df):
    assert df['date'].is_monotonic_increasing,   'Dates not sorted'
    assert df['date'].duplicated().sum() == 0,    'Duplicate dates'
    assert (df['visits'] > 0).all(),              'Zero/negative visits'
    assert df['cvr'].between(0, 100).all(),       'CVR out of range'
    assert df['bounce_rate'].between(0, 1).all(), 'Bounce rate out of range'
    print('All validations passed')


validate(unified)
unified.to_parquet('data/clean/unified_q1_2024.parquet', index=False)
unified.to_csv('data/clean/unified_q1_2024.csv', index=False)
print('Saved to data/clean/')

## Step 6 — Quick charts from the unified table

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=130)

roll_cvr = unified['cvr'].rolling(7, center=True).mean()
axes[0].plot(unified['date'], unified['cvr'], color='#4361ee', alpha=0.2)
axes[0].plot(unified['date'], roll_cvr, color='#4361ee', linewidth=2.5)
axes[0].set_title('CVR — 7d rolling', fontsize=11, fontweight='bold')
axes[0].set_ylabel('CVR (%)')

axes[1].scatter(unified['total_spend'], unified['paid_conversions'],
                color='#f72585', alpha=0.5, s=25)
axes[1].set_title('Spend vs Paid Conv.', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Daily spend ($)')
axes[1].set_ylabel('Conversions')

monthly_rev = unified.groupby('month')['revenue'].sum()
axes[2].bar(monthly_rev.index, monthly_rev.values, color='#06d6a0', edgecolor='white')
axes[2].set_title('Monthly Revenue', fontsize=11, fontweight='bold')
axes[2].set_xticks([1, 2, 3])
axes[2].set_xticklabels(['Jan', 'Feb', 'Mar'])
axes[2].set_ylabel('Revenue ($)')

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', color='#e8e8e8', linewidth=0.8)
    ax.set_axisbelow(True)

fig.suptitle('Q1 2024 — Adobe + Paid Unified View', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('data/09_etl_unified_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary — The ETL template

When you get real files from Adobe and your paid platform:

```python
adobe_clean = extract_adobe('data/raw/adobe_export.csv')
paid_clean  = extract_paid('data/raw/paid_export.csv')
paid_daily  = aggregate_paid_daily(paid_clean)
unified     = build_unified(adobe_clean, paid_daily)
validate(unified)
unified.to_parquet('data/clean/unified.parquet', index=False)
```

The only thing you need to adjust: column names inside `extract_adobe()` and `extract_paid()`. The rest stays the same.

**Next:** `10_performance.ipynb` — memory optimization, vectorization, chunking large files.